# Gemini Incremental Cache Refresh

Re-scans the Gemini `SEQUENTIAL/` and `LEAST-TO-MOST/` folders on disk and adds any newly-completed videos to:
- `all_triplets_cache.csv` — the full triplets cache
- `triplets_stratified.csv` — the stratified subset used by the judge panel

**Safe properties:**
- Existing rows keep their `row_idx` values → `panel_checkpoint.json` judge labels stay valid
- Existing stratified subset rows are preserved → no judge work is wasted
- New rows are appended at the end
- Idempotent — running twice is harmless

**Order:**
1. Run **Cell 1** (setup — defines all functions)
2. Run **Cell 2** (dry-run — reports what would change, writes nothing)
3. Run **Cell 3** (apply — actually writes the updates)
4. Re-run your existing judge-panel cell — only the new rows trigger fresh judge calls

## Cell 1 — Setup (run once)

In [1]:
import os
import re
import json
import pandas as pd
from collections import defaultdict

# =====================================================================
# PATHS
# =====================================================================
RESULTS_BASE = r'C:\Opeyemi\PROMPTS\RESULTS'
GT_PATH      = r'C:\Opeyemi\PROMPTS\ANNOTATION'
OUTPUT_DIR   = r'C:\Opeyemi\PROMPTS\EVALUATION'

GEMINI_BASE         = os.path.join(RESULTS_BASE, 'GEMINI')
GT_CACHE_PATH       = os.path.join(OUTPUT_DIR, 'gt_data_cache.json')
TRIPLETS_CACHE_PATH = os.path.join(OUTPUT_DIR, 'all_triplets_cache.csv')
STRATIFIED_PATH     = os.path.join(OUTPUT_DIR, 'triplets_stratified.csv')

# Only refresh the still-running techniques. ZERO and REACT are 807/807
# already; touching them is unnecessary risk.
RUNNING_TECHS = {
    'SEQUENTIAL':    'Sequential',
    'LEAST-TO-MOST': 'Least-to-Most',
}

# =====================================================================
# HELPERS (copied verbatim from your panel-judge notebook)
# =====================================================================
GEMINI_VID_RE = re.compile(r'^([A-Za-z]+_[A-Za-z]+\d+(?:_x264)?)')

def video_id_from_filename(filename):
    m = GEMINI_VID_RE.match(filename)
    return m.group(1) if m else None

def derive_crime_type(video_id):
    name = video_id.replace('_x264', '')
    m = re.match(r'^([A-Za-z]+?)\d', name)
    return m.group(1) if m else 'Unknown'

def is_anomalous(video_id):
    return not video_id.startswith('Normal_')

def parse_ground_truth(gt_base_path):
    GT_FILES = ['UCFCrime_Train.json', 'UCFCrime_Val.json', 'UCFCrime_Test.json']
    gt_data = {}
    for fname in GT_FILES:
        fpath = os.path.join(gt_base_path, fname)
        if not os.path.exists(fpath):
            continue
        with open(fpath, 'r', encoding='utf-8') as f:
            data = json.load(f)
        for video_id, info in data.items():
            if not is_anomalous(video_id):
                continue
            gt_data[video_id] = {
                'crime_type': derive_crime_type(video_id),
                'sentences':  info.get('sentences', []),
                'timestamps': info.get('timestamps', []),
                'duration':   info.get('duration'),
            }
    return gt_data

def load_gt():
    if os.path.exists(GT_CACHE_PATH):
        with open(GT_CACHE_PATH) as f:
            return json.load(f)
    gt = parse_ground_truth(GT_PATH)
    with open(GT_CACHE_PATH, 'w') as f:
        json.dump(gt, f)
    return gt

def resolve_gt_key(video_id, gt_data):
    candidates = [video_id, video_id.replace('_x264', '')]
    parts = video_id.split('_')
    if len(parts) >= 2:
        wp = '_'.join(parts[1:])
        candidates += [wp, wp.replace('_x264', '')]
    for c in candidates:
        if c in gt_data:
            return c
    return None

def extract_gemini_sequential_obj(obj):
    results = obj.get('sequential_results', obj)
    if not isinstance(results, dict):
        return ''
    final = results.get('Final Synthesis', {})
    if isinstance(final, dict):
        text = final.get('response', '')
        if text and text.strip():
            return text.strip()
    if isinstance(final, str) and final.strip():
        return final.strip()
    step_keys = sorted(
        [k for k in results if k.startswith('Step')],
        key=lambda x: int(x.split()[-1]) if x.split()[-1].isdigit() else 0,
        reverse=True
    )
    if step_keys:
        val = results[step_keys[0]]
        if isinstance(val, dict):
            return val.get('response', '')
        return str(val)
    return ''

def extract_gemini_ltm_obj(obj):
    results = obj.get('least_to_most_results', obj)
    if not isinstance(results, dict):
        return ''
    step_keys = sorted(
        [k for k in results if k.startswith('Step')],
        key=lambda x: int(x.split()[-1]) if x.split()[-1].isdigit() else 0,
        reverse=True
    )
    if not step_keys:
        return ''
    last = results[step_keys[0]]
    if isinstance(last, dict):
        return last.get('response', '')
    return last if isinstance(last, str) else ''

def load_gemini_running_tech(tech_folder):
    """Read all currently-completed Gemini videos for one in-progress technique."""
    tech_path = os.path.join(GEMINI_BASE, tech_folder)
    if not os.path.isdir(tech_path):
        print(f'  WARN: folder missing: {tech_path}')
        return {}

    extractor = (extract_gemini_sequential_obj if tech_folder == 'SEQUENTIAL'
                 else extract_gemini_ltm_obj)

    all_files = sorted([f for f in os.listdir(tech_path)
                        if f.endswith('.json') and '_checkpoints' not in f])
    complete  = [f for f in all_files if '_complete_' in f]

    per_video = defaultdict(list)
    for fname in complete:
        try:
            with open(os.path.join(tech_path, fname), encoding='utf-8') as f:
                obj = json.load(f)
        except Exception:
            continue
        text = extractor(obj)
        vid  = video_id_from_filename(fname)
        if vid and text and text.strip():
            per_video[vid].append(text.strip())

    return {vid: texts[-1] for vid, texts in per_video.items()}

# =====================================================================
# MAIN REFRESH FUNCTION
# =====================================================================
def refresh(do_stratified=True, dry_run=False):
    if not os.path.exists(TRIPLETS_CACHE_PATH):
        raise FileNotFoundError(
            f'{TRIPLETS_CACHE_PATH} does not exist. Run the original build first.'
        )

    print('=' * 70)
    print('Gemini incremental cache refresh' + ('  [DRY RUN]' if dry_run else ''))
    print('=' * 70)

    print('\nLoading ground truth...')
    gt_data = load_gt()
    print(f'  {len(gt_data):,} GT videos')

    print(f'\nLoading existing cache: {TRIPLETS_CACHE_PATH}')
    df_old = pd.read_csv(TRIPLETS_CACHE_PATH)
    print(f'  {len(df_old):,} rows currently')

    existing_keys = set(zip(df_old['model'], df_old['technique'], df_old['video']))

    print('\nScanning Gemini disk folders for newly-completed videos...')
    new_rows = []
    summary  = {}

    for tech_folder, tech_label in RUNNING_TECHS.items():
        outputs = load_gemini_running_tech(tech_folder)

        on_disk_count = len(outputs)
        already_in    = sum(1 for vid in outputs
                            if resolve_gt_key(vid, gt_data) is not None
                            and ('Gemini', tech_label, resolve_gt_key(vid, gt_data)) in existing_keys)

        added_for_tech = 0
        for video_id, output_text in outputs.items():
            gt_key = resolve_gt_key(video_id, gt_data)
            if gt_key is None:
                continue
            key = ('Gemini', tech_label, gt_key)
            if key in existing_keys:
                continue

            gt_info = gt_data[gt_key]
            new_rows.append({
                'model':                 'Gemini',
                'technique':             tech_label,
                'video':                 gt_key,
                'crime_type':            gt_info['crime_type'],
                'ground_truth':          ' '.join(gt_info['sentences']),
                'model_output':          output_text,
                'model_output_full_len': len(output_text),
            })
            added_for_tech += 1

        summary[tech_label] = {
            'on_disk_complete': on_disk_count,
            'already_in_cache': already_in,
            'newly_added':      added_for_tech,
        }
        print(f'  [Gemini/{tech_label}]  on disk: {on_disk_count}  '
              f'already cached: {already_in}  NEW: {added_for_tech}')

    print('\nSummary:')
    for tech, s in summary.items():
        print(f'  {tech:<15}  +{s["newly_added"]} new rows')
    print(f'  Total new rows: {len(new_rows)}')

    if not new_rows:
        print('\nNothing new to add — cache is already up to date with disk.')
        return

    if dry_run:
        print('\n[DRY RUN] Would write the new rows but dry_run=True. No files changed.')
        return

    # Append. New rows go at the end so existing row indices stay stable —
    # this is critical because PANEL_CHECKPOINT keys are "{row_idx}_{judge}".
    df_new = pd.DataFrame(new_rows)
    df_combined = pd.concat([df_old, df_new], ignore_index=True)
    df_combined.to_csv(TRIPLETS_CACHE_PATH, index=False)
    print(f'\nUpdated: {TRIPLETS_CACHE_PATH}')
    print(f'  rows: {len(df_old):,} → {len(df_combined):,}  (+{len(df_new)})')

    print('\nNew per-(model, technique) counts:')
    print(df_combined.groupby(['model', 'technique']).size().unstack(fill_value=0))

    # =================================================================
    # OPTIONAL: refresh the stratified subset
    # =================================================================
    if not do_stratified:
        print('\nSkipped stratified refresh (do_stratified=False).')
        return

    print('\n' + '-' * 70)
    print('Stratified subset refresh')
    print('-' * 70)

    N_PER_STRATUM = 5
    RANDOM_SEED   = 42

    if os.path.exists(STRATIFIED_PATH):
        df_strat_old = pd.read_csv(STRATIFIED_PATH)
        print(f'\n  Existing stratified subset: {len(df_strat_old):,} rows')
    else:
        df_strat_old = pd.DataFrame(columns=df_combined.columns)
        print(f'\n  No existing stratified subset — creating fresh.')

    strat_existing_keys = set(zip(df_strat_old['model'],
                                  df_strat_old['technique'],
                                  df_strat_old['video']))

    candidates = df_combined[
        (df_combined['model'] == 'Gemini')
        & (df_combined['technique'].isin(RUNNING_TECHS.values()))
    ]
    candidates = candidates[~candidates.apply(
        lambda r: (r['model'], r['technique'], r['video']) in strat_existing_keys, axis=1
    )]

    topup_rows = []
    for tech_label in RUNNING_TECHS.values():
        for crime in df_combined['crime_type'].unique():
            already = df_strat_old[
                (df_strat_old['model'] == 'Gemini')
                & (df_strat_old['technique'] == tech_label)
                & (df_strat_old['crime_type'] == crime)
            ]
            need = N_PER_STRATUM - len(already)
            if need <= 0:
                continue
            pool = candidates[
                (candidates['technique'] == tech_label)
                & (candidates['crime_type'] == crime)
            ]
            if len(pool) == 0:
                continue
            pick = pool.sample(n=min(need, len(pool)), random_state=RANDOM_SEED)
            topup_rows.append(pick)

    if topup_rows:
        df_topup = pd.concat(topup_rows, ignore_index=True)
        df_strat_new = pd.concat([df_strat_old, df_topup], ignore_index=True)
        df_strat_new.to_csv(STRATIFIED_PATH, index=False)
        print(f'\n  Topped up subset: {len(df_strat_old):,} → {len(df_strat_new):,} '
              f'(+{len(df_topup)})')
        print('\n  New stratified per-(model, technique):')
        print(df_strat_new.groupby(['model', 'technique']).size().unstack(fill_value=0))
    else:
        print('\n  No top-up needed — all Gemini Sequential / LtM strata are at target N.')

    print('\nReady. You can re-run the panel-judge cell now;')
    print('the existing panel_checkpoint.json will be honoured for unchanged rows,')
    print('and only the newly-added rows will trigger fresh judge calls.')

print('Setup complete. Functions ready: refresh(do_stratified=True, dry_run=False)')

Setup complete. Functions ready: refresh(do_stratified=True, dry_run=False)


## Cell 2 — Dry run (preview only, writes nothing)

Run this first to see exactly what would change. It will tell you how many newly-completed Gemini videos are on disk that aren't in the cache yet.

In [2]:
refresh(do_stratified=True, dry_run=True)

Gemini incremental cache refresh  [DRY RUN]

Loading ground truth...
  944 GT videos

Loading existing cache: C:\Opeyemi\PROMPTS\EVALUATION\all_triplets_cache.csv
  8,900 rows currently

Scanning Gemini disk folders for newly-completed videos...
  [Gemini/Sequential]  on disk: 678  already cached: 674  NEW: 0
  [Gemini/Least-to-Most]  on disk: 160  already cached: 159  NEW: 0

Summary:
  Sequential       +0 new rows
  Least-to-Most    +0 new rows
  Total new rows: 0

Nothing new to add — cache is already up to date with disk.


## Cell 3 — Apply the update

If the dry run looks right, run this to actually write the new rows to `all_triplets_cache.csv` and top up `triplets_stratified.csv`.

In [3]:
refresh(do_stratified=True, dry_run=False)

Gemini incremental cache refresh

Loading ground truth...
  944 GT videos

Loading existing cache: C:\Opeyemi\PROMPTS\EVALUATION\all_triplets_cache.csv
  8,900 rows currently

Scanning Gemini disk folders for newly-completed videos...
  [Gemini/Sequential]  on disk: 678  already cached: 674  NEW: 0
  [Gemini/Least-to-Most]  on disk: 160  already cached: 159  NEW: 0

Summary:
  Sequential       +0 new rows
  Least-to-Most    +0 new rows
  Total new rows: 0

Nothing new to add — cache is already up to date with disk.


In [4]:
import pandas as pd

df = pd.read_csv(r'C:\Opeyemi\PROMPTS\EVALUATION\triplets_stratified.csv')

print(f'Total: {len(df)} rows')
print()
print('Per (model, technique):')
print(df.groupby(['model', 'technique']).size().unstack(fill_value=0))
print()
print('Gemini Sequential coverage by crime type:')
print(df[(df['model']=='Gemini') & (df['technique']=='Sequential')]
      .groupby('crime_type').size())
print()
print('Gemini Least-to-Most coverage by crime type:')
print(df[(df['model']=='Gemini') & (df['technique']=='Least-to-Most')]
      .groupby('crime_type').size())

Total: 616 rows

Per (model, technique):
technique  Least-to-Most  ReAct  Sequential  Zero-Shot
model                                                 
Claude                55     55          55         55
GPT                   55     55          55         55
Gemini                16     55          50         55

Gemini Sequential coverage by crime type:
crime_type
Abuse            5
Assault          5
Burglary         5
Explosion        5
Fighting         5
RoadAccidents    5
Robbery          5
Shooting         5
Shoplifting      5
Stealing         5
dtype: int64

Gemini Least-to-Most coverage by crime type:
crime_type
Abuse        5
Assault      5
Burglary     5
Explosion    1
dtype: int64


## Cell 4 — (Optional) Update only the triplets cache, skip stratified resampling

Use this variant if you want to control the stratified subset manually. It only updates `all_triplets_cache.csv`.

In [ ]:
# Uncomment to run this variant instead of Cell 3:
# refresh(do_stratified=False, dry_run=False)

---

## After running

Re-run your existing **judge-panel cell**. Because:
- Existing rows kept their `row_idx` values
- `panel_checkpoint.json` keys are `{row_idx}_{judge}`

...all your old judge labels will be reused, and only the newly-added Gemini rows will trigger fresh API calls.

Run this notebook again whenever your Gemini generation produces more `_complete_` files.